In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electronics_retailer_clg.silver;

In [0]:
data = spark.table("electronics_retailer_clg.bronze.stores")

In [0]:

import pyspark.sql.functions as F

def standardize_date(columnName, df):
    return df.withColumn(
        "date_parsed",
        F.coalesce(
            F.try_to_date(F.col(columnName), "M/d/yyyy"),
            F.try_to_date(F.col(columnName), "M-d-yyyy"),
            F.try_to_date(F.col(columnName), "yyyy-M-d"),
            F.try_to_date(F.col(columnName), "yyyy/M/d")
        )
    ).withColumn(
        columnName,
        F.trim(F.col("date_parsed")).cast('date')
    ).drop("date_parsed")

data = standardize_date("open_date", data)
display(data)

In [0]:

def change_dataType(df,dataType,column):
    df = df.withColumn(column,df[column].cast(dataType))
    return df
data = change_dataType(data,"int","storekey")
data = change_dataType(data,"int","square_meters")
display(data)

In [0]:
data.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("electronics_retailer_clg.silver.stores")

print("Store cleaned successfully")

In [0]:
# from pyspark.sql.functions import col, trim

# df = spark.table("electronics_retailer_clg.bronze.stores")


# df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])


# for c in df.columns:
#     df = df.withColumn(c, trim(col(c)))



# df = df.withColumn("storekey", col("storekey").cast("int"))


# df = df.fillna({
#     "country": "unknown"
# })


# df = df.select(
#     "storekey",
#     "country"
# )


# df = df.dropDuplicates(["storekey"])



# display(df)
# df.printSchema()


# df.write.format("delta") \
#     .mode("overwrite") \
#     .option("mergeSchema", "true") \
#     .saveAsTable("electronics_retailer_clg.silver.stores")

# print("Store cleaned successfully")